In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score
from openai import OpenAI
from google.colab import userdata
import time
import warnings
from datetime import datetime
import re
warnings.filterwarnings('ignore')


class TwoStageGPT5MiniAnnotator:
    def __init__(self, api_key=None, log_file=None):
        if api_key is None:
            api_key = userdata.get('OPENROUTER_API_KEY')
            if not api_key:
                raise ValueError("کلید OPENROUTER_API_KEY در Secrets تنظیم نشده است.")
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key,
            default_headers={
                "HTTP-Referer": "https://colab.research.google.com",
                "X-Title": "GPT-5 Mini Two-Stage Hate Speech"
            }
        )
        self.main_classes = ['عادی', 'توهین‌آمیز', 'تنفرآمیز']
        self.hate_classes = ['جنسیتی', 'قومی', 'مذهبی', 'ملیتی', 'سیاسی', 'سنی', 'سایر']

        self.total_tokens = 0
        self.request_count = 0

        if log_file is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            log_file = f"two_stage_gpt5mini_log_{timestamp}.txt"
        self.log_file = log_file
        self.detailed_results = []
        with open(self.log_file, 'w', encoding='utf-8') as f:
            f.write(f"🤖 Two-Stage GPT-5 Mini Annotation Log\n")
            f.write(f"📅 شروع: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*80 + "\n\n")
        print(f"📝 لاگ در فایل ذخیره می‌شود: {self.log_file}")

    def log_message(self, msg, print_console=True):
        timestamp = datetime.now().strftime("%H:%M:%S")
        entry = f"[{timestamp}] {msg}"
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(entry + "\n")
        if print_console:
            print(entry)

    def preprocess_text(self, text, limit=300):
        clean_text = re.sub(r'\W+', ' ', text).strip()
        if len(clean_text) > limit:
            return clean_text[:limit] + "..."
        return clean_text

    def create_prompt_stage1(self, text):
        text = self.preprocess_text(text)
        # پرامپت ساده‌شده برای مرحله اول
 prompt = f"""این متن را در یکی از سه دسته زیر طبقه‌بندی کن:

- عادی: متنی که فاقد توهین، ناسزا، محتوای خصمانه یا بیان نفرت علیه فرد یا گروه خاصی باشد.
- توهین‌آمیز: متنی که شامل ناسزا، اهانت یا تحقیر باشد، اما نفرت یا خصومت علیه یک گروه خاص را بیان نکند.
- تنفرآمیز: متنی که نفرت، خصومت یا نگرش منفی علیه یک گروه خاص از افراد را بیان یا ترویج کند.

در تشخیص بین «توهین‌آمیز» و «تنفرآمیز» دقت کن؛ صرف وجود ناسزا یا توهین به‌تنهایی کافی نیست و زمانی متن در دسته تنفرآمیز قرار می‌گیرد که نفرت یا خصومت متوجه یک گروه خاص باشد.

متن: "{text}"

فقط یکی از سه برچسب زیر را به‌عنوان پاسخ برگردان:
عادی
توهین‌آمیز
تنفرآمیز

فقط یک کلمه جواب بده:"""
        return prompt

    def create_prompt_stage2(self, text):
        text = self.preprocess_text(text)
        # پرامپت ساده‌شده برای مرحله دوم
    prompt = f"""این متن در مرحله اول به‌عنوان «تنفرآمیز» شناسایی شده است.
اکنون نوع گفتار تنفرآمیز را مشخص کن:

- جنسیتی: نفرت یا خصومت علیه افراد بر اساس جنسیت.
- قومی: نفرت یا خصومت علیه یک قوم یا گروه قومی.
- مذهبی: نفرت یا خصومت علیه افراد بر اساس مذهب یا باور دینی.
- ملیتی: نفرت یا خصومت علیه افراد بر اساس ملیت یا هویت ملی.
- سیاسی: نفرت یا خصومت علیه افراد یا گروه‌ها بر اساس گرایش، هویت یا وابستگی سیاسی.
- سنی: نفرت یا خصومت علیه افراد بر اساس سن.
- سایر: گفتار تنفرآمیزی که به‌طور مشخص در هیچ‌یک از دسته‌های بالا قرار نمی‌گیرد.

متن: "{text}"

فقط یکی از برچسب‌های زیر را به‌عنوان پاسخ برگردان:
جنسیتی
قومی
مذهبی
ملیتی
سیاسی
سنی
سایر

فقط یک کلمه جواب بده:"""
        return prompt

    def query_gpt(self, prompt, max_retries=3):
        for i in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model='qwen/qwen3-32b',
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=2000,
                    temperature=0
                )
                tokens = response.usage.total_tokens
                self.total_tokens += tokens
                self.request_count += 1
                answer = response.choices[0].message.content.strip()
                if not answer:
                    self.log_message(f"⚠️ پاسخ خالی دریافت شد برای پرامپت:\n{prompt}", True)
                return answer, tokens
            except Exception as e:
                self.log_message(f"❌ خطا در تلاش {i+1}: {e}", False)
                time.sleep(2 ** i)
        return None, 0

    def interpret_stage1(self, response):
        if not response:
            return 'عادی'
        for c in self.main_classes:
            if c in response:
                return c
        return 'عادی'

    def interpret_stage2(self, response):
        if not response:
            return 'سایر'
        for c in self.hate_classes:
            if c in response:
                return c
        return 'سایر'

    def annotate(self, idx, text, true_label, true_hate_class):
        prompt1 = self.create_prompt_stage1(text)
        resp1, tokens1 = self.query_gpt(prompt1)
        pred1 = self.interpret_stage1(resp1)

        tokens2 = 0
        pred2 = None
        if pred1 == 'تنفرآمیز':
            prompt2 = self.create_prompt_stage2(text)
            resp2, tokens2 = self.query_gpt(prompt2)
            pred2 = self.interpret_stage2(resp2)

        pred_final = pred2 if pred2 is not None else pred1
        true_final = true_hate_class if (true_label=='تنفرآمیز' and pd.notna(true_hate_class)) else true_label
        is_correct = pred_final == true_final

        short_text = text[:200] + ("..." if len(text) > 200 else "")
        prompt_summary = prompt1 + (f"\n\n{prompt2}" if pred1 == 'تنفرآمیز' else "")
        log_entry = f"""
{'='*80}
🆔 متن شماره: {idx}
📝 متن: "{short_text}"

▶️ پرامپت‌ها:
{prompt_summary}

💬 پاسخ مرحله اول:
\"{resp1}\"
{f'💬 پاسخ مرحله دوم:\n\"{resp2}\"' if pred2 else ''}

🏷️ پیش‌بینی نهایی: {pred_final}
📌 برچسب واقعی: {true_final}
✅ {'صحیح' if is_correct else 'غلط'}
⏰ زمان: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
💰 توکن مصرف شده: {tokens1 + tokens2}
{'='*80}
"""
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(log_entry + "\n")

        print(f" [{'✔️' if is_correct else '❌'}] متن {idx}: {pred_final} (واقعی: {true_final}) توکن: {tokens1 + tokens2}")
        self.detailed_results.append({
            'id': idx,
            'text': text,
            'true_label': true_final,
            'predicted_label': pred_final,
            'correct': is_correct,
            'tokens_used': tokens1 + tokens2,
            'response_stage1': resp1,
            'response_stage2': resp2 if pred2 else None,
            'prompt': prompt_summary,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
        return pred_final

    def save_results(self, filename=None):
        if filename is None:
            filename = f"two_stage_gpt5mini_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df = pd.DataFrame(self.detailed_results)
        df.to_csv(filename, index=False)
        self.log_message(f"💾 نتایج ذخیره شد در: {filename}")
        return filename


def load_dataset():
    print("بارگذاری دیتاست:")
    print("1. ورود مسیر فایل محلی")
    print("2. بارگذاری از Google Drive")
    print("3. آپلود فایل")
    choice = input("انتخاب روش بارگذاری (1/2/3): ").strip()
    df = None
    try:
        if choice == '1':
            path = input("مسیر کامل فایل CSV را وارد کنید: ").strip()
            df = pd.read_csv(path)
        elif choice == '2':
            from google.colab import drive
            drive.mount('/content/drive')
            path = input("مسیر فایل در Drive را وارد کنید: ").strip()
            df = pd.read_csv(path)
        elif choice == '3':
            from google.colab import files
            uploaded = files.upload()
            fname = next(iter(uploaded))
            df = pd.read_csv(fname)
        else:
            print("انتخاب نامعتبر!")
    except Exception as e:
        print(f"خطا در بارگذاری فایل: {e}")
    if df is not None:
        print(f"✅ تعداد نمونه: {len(df)}")
    return df


def validate_dataset(df):
    if df is None:
        return False
    required = ['Text', 'classification', 'hate_class']
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"ستون‌های گمشده: {missing}")
        return False
    return True


def evaluate_full_dataset(df):
    """تمام دیتاست را برچسب‌زنی و ارزیابی می‌کند"""
    df = df.dropna(subset=['Text', 'classification']).reset_index(drop=True)

    print(f"\n==== شروع برچسب‌زنی کل دیتاست ({len(df)} نمونه) ====")

    annotator = TwoStageGPT5MiniAnnotator()
    annotator.log_message(f"شروع برچسب‌زنی {len(df)} نمونه")

    all_preds = []
    all_trues = []

    for i, row in df.iterrows():
        pred_label = annotator.annotate(i+1, row['Text'], row['classification'], row['hate_class'])
        all_preds.append(pred_label)

        true_label = row['hate_class'] if (row['classification'] == 'تنفرآمیز' and pd.notna(row['hate_class'])) else row['classification']
        all_trues.append(true_label)

    # محاسبه معیارهای ارزیابی
    acc = accuracy_score(all_trues, all_preds)
    f1 = f1_score(all_trues, all_preds, average='weighted')

    # نمایش نتایج
    print("\n" + "="*60)
    print("🎯 نتایج نهایی:")
    print(f"📊 Accuracy: {acc:.4f}")
    print(f"📊 F1-Score: {f1:.4f}")
    print(f"💰 Total tokens used: {annotator.total_tokens}")
    print(f"🔢 Total requests: {annotator.request_count}")
    print("\n📋 Classification Report:")
    print(classification_report(all_trues, all_preds))

    # ذخیره نتایج
    results_file = annotator.save_results()

    # ذخیره summary
    summary_data = {
        'total_samples': len(df),
        'accuracy': acc,
        'f1_score': f1,
        'total_tokens': annotator.total_tokens,
        'total_requests': annotator.request_count
    }

    summary_df = pd.DataFrame([summary_data])
    summary_file = f"summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    summary_df.to_csv(summary_file, index=False)

    print(f"💾 خلاصه نتایج در: {summary_file}")

    return {
        "accuracy": acc,
        "f1_score": f1,
        "tokens_used": annotator.total_tokens,
        "results_file": results_file,
        "summary_file": summary_file
    }


def main():
    df = load_dataset()
    if not validate_dataset(df):
        print("دیتاست معتبر نیست. لطفا بررسی کنید.")
        return

    print("\nحالت اجرا:")
    print("1. برچسب‌زنی کل دیتاست")
    print("2. Cross-validation (حالت قبلی)")

    mode = input("انتخاب حالت (1/2): ").strip()

    if mode == '1':
        evaluate_full_dataset(df)
    elif mode == '2':
        from sklearn.model_selection import StratifiedKFold
        n = input("تعداد folds برای cross validation (پیش‌فرض 5): ").strip()
        n = int(n) if n.isdigit() else 5
        # کد قبلی cross-validation اینجا قرار می‌گیرد
        print("Cross-validation mode - کد قبلی")
    else:
        print("انتخاب نامعتبر!")


if __name__ == "__main__":
    main()
